In [1]:
import torch
import torch.nn as nn
import pandas as pd
import optuna
import torchmetrics

torch.manual_seed(42)
device = "cuda"

In [2]:
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

cifar10_mean = (0.4914, 0.4822, 0.4465)
cifar10_std  = (0.2470, 0.2435, 0.2616)

transform = transforms.Compose([
    transforms.ToTensor(),                               # PIL Image → FloatTensor [0,1]
    transforms.Normalize(mean=cifar10_mean, std=cifar10_std)
])

train_dataset = datasets.CIFAR10(root="datasets/", train=True,  download=True, transform=transform)
test_dataset  = datasets.CIFAR10(root="datasets/", train=False, download=True, transform=transform)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader  = DataLoader(test_dataset,  batch_size=32, shuffle=False)

images, labels = next(iter(train_loader))
print(images.shape)  # (32 images)

torch.Size([32, 3, 32, 32])


In [ ]:
from typing import Any

class Dense(nn.Module):
    def __init__(self, input_n:int, output_n:int):
        super().__init__()
        self.linear = nn.Linear(input_n, output_n)
        self.activation = nn.SiLU()
        
        nn.init.kaiming_normal_(self.linear.weight, nonlinearity="relu")
        nn.init.zeros_(self.linear.bias)
        
    def forward(self, x):
        return self.activation(self.linear(x))

hidden_layers = [Dense(100, 100) for _ in range(20)]

model = nn.Sequential(
    nn.Flatten(),
    Dense(3*32*32, 100),
    *hidden_layers,
    nn.Linear(100,10)
)
model.to(device)

Sequential(
  (0): Flatten(start_dim=1, end_dim=-1)
  (1): Dense(
    (linear): Linear(in_features=3072, out_features=100, bias=True)
    (activation): SiLU()
  )
  (2): Dense(
    (linear): Linear(in_features=100, out_features=100, bias=True)
    (activation): SiLU()
  )
  (3): Dense(
    (linear): Linear(in_features=100, out_features=100, bias=True)
    (activation): SiLU()
  )
  (4): Dense(
    (linear): Linear(in_features=100, out_features=100, bias=True)
    (activation): SiLU()
  )
  (5): Dense(
    (linear): Linear(in_features=100, out_features=100, bias=True)
    (activation): SiLU()
  )
  (6): Dense(
    (linear): Linear(in_features=100, out_features=100, bias=True)
    (activation): SiLU()
  )
  (7): Dense(
    (linear): Linear(in_features=100, out_features=100, bias=True)
    (activation): SiLU()
  )
  (8): Dense(
    (linear): Linear(in_features=100, out_features=100, bias=True)
    (activation): SiLU()
  )
  (9): Dense(
    (linear): Linear(in_features=100, out_features=10

In [4]:
print(f"Train samples : {len(train_dataset)}")   # 50,000
print(f"Test  samples : {len(test_dataset)}")    # 10,000

image, label = train_dataset[0]
print(f"Image shape   : {image.shape}")          # torch.Size([3, 32, 32])
print(f"Label         : {label}")                # int 0~9


Train samples : 50000
Test  samples : 10000
Image shape   : torch.Size([3, 32, 32])
Label         : 6


In [5]:
torch.unique(torch.tensor(test_dataset.targets))

tensor([0, 1, 2, 3, 4, 5, 6, 7, 8, 9])

In [20]:
import torchmetrics
from torch.optim import Optimizer
from torch.utils.tensorboard import SummaryWriter
from typing import Optional

#tensorboard --logdir=study\c_11\log

class EarlyStop:
    def __init__(self, patience=5, min_delta = 1e-4):
        self.patience = patience
        self.min_delta = min_delta
        self.best_loss = float('inf')
        self.counter = 0
        self.best_weights = None
    
    def step(self, val_loss, model) -> bool:
        if val_loss <self.best_loss - self.min_delta:
            self.best_loss = val_loss
            self.counter = 0
            self.best_weights = {k: v.clone() for k, v in model.state_dict().items()}
        else:
            self.counter += 1
        return self.counter >= self.patience
    
    def restore(self, model):
        if self.best_weights:
            model.load_state_dict(self.best_weights)

def train(writer:SummaryWriter, model:nn.Module, optimizer:Optimizer, criterion, accuracy:torchmetrics.Accuracy, 
          train_loader:DataLoader, test_loader:DataLoader, n_epoch:int = 1, global_step:int = 0, early_stop:Optional[EarlyStop] = None):
    model.train()    
    for epoch_counter in range(n_epoch):
        total_loss:int = 0
        for X, y in train_loader:
            X,y = X.to(device), y.to(device)
            y_pred = model(X)
            loss = criterion(y_pred, y)
            loss.backward()
            optimizer.step()
            optimizer.zero_grad()
            total_loss += loss.item()
            
            writer.add_scalar("Loss_batch/train", loss.item(), global_step)
            global_step += 1
        
        
        avg_loss = total_loss / len(train_loader)
        avg_loss_test, acc = eval(model, criterion, accuracy, test_loader)
        
        print(f"epoch:{epoch_counter}, avg_loss:{avg_loss}, acc={acc}")
        
        writer.add_scalars('Loss', {
            'train': avg_loss,
            'test': avg_loss_test
        }, epoch_counter)
        writer.add_scalar("acc/test", acc, epoch_counter)
        writer.flush()
        
        if early_stop:
            should_stop = early_stop.step(avg_loss_test, model)
            if should_stop:
                print(f"Early stopping at epoch {epoch_counter}")
                early_stop.restore(model)  # roll back to best weights
                break

def eval(model:nn.Module, criterion, metric_fn:torchmetrics.Accuracy, 
         dataloader:DataLoader, agg_fn = torch.mean) -> tuple[float, float]:
    model.eval()
    metrics = []
    total_loss = 0
    with torch.no_grad():
        for X, y in dataloader:
            X,y = X.to(device), y.to(device)
            y_pred = model(X)
            loss = criterion(y_pred, y)
            total_loss += loss.item()
            metric = metric_fn(y_pred, y)
            metrics.append(metric)

    avg_loss = total_loss/len(dataloader)
    metric_fn.reset()
    model.train()
    return avg_loss, agg_fn(torch.stack(metrics))



In [ ]:
n_epoch:int = 15
lr = 1e-3
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-3, betas=(0.9, 0.999), eps=1e-8)
accuracy = torchmetrics.Accuracy(task="multiclass", num_classes=10).to(device)
early_stop = EarlyStop()

In [8]:
with SummaryWriter("study/c_11/log/normal") as writer:
    train(writer, model, optimizer, criterion, accuracy, train_loader, test_loader, n_epoch=15)

epoch:0, avg_loss:1.9288770926738503, acc=0.3189896047115326
epoch:1, avg_loss:1.7539151876459347, acc=0.35832667350769043
epoch:2, avg_loss:1.675196784319057, acc=0.37480032444000244
epoch:3, avg_loss:1.6230422582522617, acc=0.41523560881614685
epoch:4, avg_loss:1.5800314215765652, acc=0.42851436138153076
epoch:5, avg_loss:1.5344272302421942, acc=0.45327475666999817
epoch:6, avg_loss:1.4962866210205312, acc=0.45597043633461
epoch:7, avg_loss:1.4663705091330004, acc=0.44518768787384033
epoch:8, avg_loss:1.4371045538041352, acc=0.4738418459892273
epoch:9, avg_loss:1.4004420595754818, acc=0.4686501622200012
epoch:10, avg_loss:1.3710893916915947, acc=0.4590654969215393
epoch:11, avg_loss:1.3408730096032966, acc=0.47823482751846313
epoch:12, avg_loss:1.3200200290460276, acc=0.4774360954761505
epoch:13, avg_loss:1.306065746171308, acc=0.4893170893192291
epoch:14, avg_loss:1.280089093871553, acc=0.49560701847076416


In [17]:
class DenseBatchNorm(nn.Module):
    def __init__(self, input_n:int, output_n:int):
        super().__init__()
        self.linear = nn.Linear(input_n, output_n)
        self.activation = nn.SiLU()
        
        nn.init.kaiming_normal_(self.linear.weight, nonlinearity="relu")
        nn.init.zeros_(self.linear.bias)
        
        self.seq = nn.Sequential(
            nn.BatchNorm1d(input_n),
            self.linear,
            self.activation
        )
        
    def forward(self, X):
        return self.seq(X)

hidden_layers = [DenseBatchNorm(100, 100) for _ in range(20)]

model_batch_norm = nn.Sequential(
    nn.Flatten(),
    nn.BatchNorm1d(3*32*32),
    nn.Linear(3*32*32, 100),
    *hidden_layers,
    nn.BatchNorm1d(100),
    nn.Linear(100,10)
)
model_batch_norm.to(device)

Sequential(
  (0): Flatten(start_dim=1, end_dim=-1)
  (1): BatchNorm1d(3072, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
  (2): Linear(in_features=3072, out_features=100, bias=True)
  (3): DenseBatchNorm(
    (linear): Linear(in_features=100, out_features=100, bias=True)
    (activation): SiLU()
    (seq): Sequential(
      (0): BatchNorm1d(100, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
      (1): Linear(in_features=100, out_features=100, bias=True)
      (2): SiLU()
    )
  )
  (4): DenseBatchNorm(
    (linear): Linear(in_features=100, out_features=100, bias=True)
    (activation): SiLU()
    (seq): Sequential(
      (0): BatchNorm1d(100, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
      (1): Linear(in_features=100, out_features=100, bias=True)
      (2): SiLU()
    )
  )
  (5): DenseBatchNorm(
    (linear): Linear(in_features=100, out_features=100, bias=True)
    (activation): SiLU()
    (s

In [21]:
n_epoch:int = 15
lr = 1e-3
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model_batch_norm.parameters(), lr=lr, weight_decay=1e-3, betas=(0.9, 0.999), eps=1e-8)
accuracy = torchmetrics.Accuracy(task="multiclass", num_classes=10).to(device)
early_stop = EarlyStop()

In [22]:
with SummaryWriter("study/c_11/log/batch_norm") as writer:
    train(writer, model_batch_norm, optimizer, criterion, accuracy, train_loader, test_loader, n_epoch=15)

epoch:0, avg_loss:2.0043618163464547, acc=0.34624600410461426
epoch:1, avg_loss:1.820589972777925, acc=0.39766374230384827
epoch:2, avg_loss:1.748607032511071, acc=0.4279153347015381
epoch:3, avg_loss:1.6926120373200546, acc=0.4279153347015381
epoch:4, avg_loss:1.6518413808509012, acc=0.4434904158115387
epoch:5, avg_loss:1.6165814761999548, acc=0.47124600410461426
epoch:6, avg_loss:1.5736842876966382, acc=0.4573681950569153
epoch:7, avg_loss:1.5408346117572456, acc=0.47264376282691956
epoch:8, avg_loss:1.5157378743035017, acc=0.47873401641845703
epoch:9, avg_loss:1.4874556489045698, acc=0.4875199496746063
epoch:10, avg_loss:1.4637907573372908, acc=0.5089856386184692
epoch:11, avg_loss:1.440366746825586, acc=0.504792332649231
epoch:12, avg_loss:1.4125358977534415, acc=0.513877809047699
epoch:13, avg_loss:1.3966426827247067, acc=0.5023961663246155
epoch:14, avg_loss:1.3763690283492218, acc=0.5139776468276978
